# 📱 Host game mobile trên Google Colab / Kaggle

Chạy **Android thật** trong notebook, điều khiển từ điện thoại qua trình duyệt.

| | |
|---|---|
| **Có `/dev/kvm`** | Dùng **Android Emulator** chính chủ + ảnh **Google Play** (API 30/33) → có Play Store, cài app lậu thoải mái, hiệu năng gần thật |
| **Không có `/dev/kvm`** (Kaggle, Colab free) | Dùng **QEMU + android-x86_64 ISO** chạy TCG (mô phỏng mềm) → vẫn boot được nhưng chậm hơn nhiều |

Điều khiển từ điện thoại: màn hình Android được stream về web, chạm vào màn hình = tap trên máy Android.

> ⚠️ Colab/Kaggle **cắt session khi không hoạt động** và giới hạn thời gian. Đây là máy chơi tạm, không phải máy chơi lâu dài.


In [ ]:
# ══════════ CELL 1 · Kiểm tra môi trường ══════════
import os, sys, platform, subprocess, shutil

KVM = os.path.exists("/dev/kvm")
ARCH = platform.machine()
print("python      :", sys.version.split()[0])
print("arch        :", ARCH)
print("/dev/kvm    :", "CÓ ✅" if KVM else "KHÔNG ❌  → sẽ dùng QEMU (mô phỏng mềm, chậm hơn)")
print("cpu count   :", os.cpu_count())
print("disk free   :", shutil.disk_usage("/").free // (1024**3), "GB")

try:
    mem = subprocess.run(["free","-g"],capture_output=True,text=True).stdout
    print("RAM total   :", [l for l in mem.splitlines() if l.startswith("Mem:")][0].split()[1], "GB")
except Exception:
    pass

MODE = "emulator" if KVM else "qemu"
print("\n>>> CHẾ ĐỘ:", MODE.upper())


## 📦 CELL 2 · Cài đặt công cụ

Tải **platform-tools** (adb) và **QEMU** (đi kèm trong gói emulator của Google, nên chỉ tải 1 lần cho cả 2 chế độ).

In [ ]:
# ══════════ CELL 2 · Cài đặt adb + QEMU ══════════
import os, subprocess, urllib.request, zipfile, sys, stat, json, shutil

ROOT = "/content/android"
os.makedirs(ROOT, exist_ok=True)
os.chdir(ROOT)

def sh(cmd, **kw):
    return subprocess.run(cmd, shell=isinstance(cmd,str), capture_output=True, text=True, **kw)

def dl(url, dest, quiet=False):
    if not quiet: print("↓", url.split("/")[-1], "…", flush=True)
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=120) as r, open(dest,"wb") as f:
        shutil.copyfileobj(r, f)
    return dest

# ── 1) platform-tools (adb) ──────────────────────────────
if not os.path.exists(f"{ROOT}/platform-tools/adb"):
    dl("https://dl.google.com/android/repository/platform-tools-latest-linux.zip", "pt.zip")
    with zipfile.ZipFile("pt.zip") as z: z.extractall(ROOT)
    os.remove("pt.zip")
os.environ["PATH"] = f"{ROOT}/platform-tools:" + os.environ["PATH"]
print("adb:", subprocess.run(["adb","version"],capture_output=True,text=True).stdout.strip().splitlines()[0])


## 🧠 CELL 3 · Boot Android

- **Có KVM** → Android Emulator + ảnh Google Play (có Play Store).
- **Không KVM** → QEMU chạy `android-x86_64` ISO. Đây là bản **không kèm Play Store**; cài app bằng APK (adb install) hoặc dùng AOSP build có Play.

Chạy ô này và **chờ** đến khi in ra `BOOT_OK`.

In [ ]:
# ══════════ CELL 3 · Boot Android ══════════
import os, subprocess, urllib.request, zipfile, shutil, time, sys, re

ROOT = "/content/android"
os.chdir(ROOT)

def sh(cmd, **kw):
    return subprocess.run(cmd, shell=isinstance(cmd,str), capture_output=True, text=True, **kw)

def dl(url, dest):
    print("↓", url.split("/")[-1], "…", flush=True)
    req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=180) as r, open(dest,"wb") as f:
        shutil.copyfileobj(r, f)

EMULATOR_URL = "https://dl.google.com/android/repository/emulator-linux_x64-16982698.zip"

def ensure_qemu():
    """QEMU binary đi kèm gói emulator của Google."""
    need = ["emulator/qemu/linux-x86_64/qemu-system-x86_64"]
    p = f"{ROOT}/emulator"
    if os.path.exists(f"{p}/qemu-system-x86_64"):
        return p
    if not os.path.exists("emulator-linux.zip"):
        dl(EMULATOR_URL, "emulator-linux.zip")
    print("  giải nén emulator…", flush=True)
    with zipfile.ZipFile("emulator-linux.zip") as z:
        for n in z.namelist():
            if n.startswith("emulator/qemu/linux-x86_64/qemu-system-x86_64"):
                z.extract(n, ROOT)
                break
    os.rename(f"{p}/qemu/linux-x86_64/qemu-system-x86_64", f"{p}/qemu-system-x86_64")
    shutil.rmtree(f"{p}/qemu", ignore_errors=True)
    os.chmod(f"{p}/qemu-system-x86_64", os.stat(f"{p}/qemu-system-x86_64").st_mode | stat.S_IEXEC)
    return p

def adb_ready(timeout=900):
    """Chờ Android boot xong (sys.boot_completed=1)."""
    t0 = time.time()
    while time.time() - t0 < timeout:
        r = sh(["adb","devices"])
        if "device" in r.stdout and "unauthorized" not in r.stdout:
            b = sh(["adb","shell","getprop","sys.boot_completed"]).stdout.strip()
            if b == "1":
                return True
        time.sleep(5)
    return False

# ═══════════════════════════════════════════════════════════
if MODE == "emulator":
    # ── Có KVM: Android Emulator + Google Play ──────────────
    import glob
    sdk = f"{ROOT}/sdk"
    if not os.path.exists(f"{sdk}/cmdline-tools/latest/bin/sdkmanager"):
        os.makedirs(f"{sdk}/cmdline-tools", exist_ok=True)
        dl("https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip", "clt.zip")
        with zipfile.ZipFile("clt.zip") as z: z.extractall(f"{sdk}/cmdline-tools")
        os.rename(f"{sdk}/cmdline-tools/cmdline-tools", f"{sdk}/cmdline-tools/latest")
        os.remove("clt.zip")
    if not shutil.which("java"):
        print("  cài java…", flush=True)
        sh("apt-get update -qq && apt-get install -y -qq default-jdk-headless")
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/default-java"
    os.environ["ANDROID_SDK_ROOT"] = sdk
    os.environ["ANDROID_HOME"] = sdk

    IMG = "system-images;android-30;google_apis_playstore;x86_64"
    if not os.path.exists(f"{sdk}/system-images/android-30/google_apis_playstore/x86_64"):
        print("  tải ảnh Google Play (~1.5GB)…", flush=True)
        sh(f"yes | {sdk}/cmdline-tools/latest/bin/sdkmanager --sdk_root={sdk} '{IMG}'")
    if not os.path.exists(f"{ROOT}/avd/phone.avd"):
        sh(f"echo no | {sdk}/cmdline-tools/latest/bin/avdmanager create avd -n phone -k '{IMG}' --force")
    os.makedirs(f"{ROOT}/avd", exist_ok=True)

    os.makedirs(f"{sdk}/emulator", exist_ok=True)
    if not os.path.exists(f"{sdk}/emulator/emulator"):
        print("  tải emulator…", flush=True)
        dl(EMULATOR_URL, "emulator-linux.zip")
        with zipfile.ZipFile("emulator-linux.zip") as z:
            for n in z.namelist():
                if n.startswith("emulator/") and not n.endswith("/"):
                    z.extract(n, sdk)
        os.remove("emulator-linux.zip")

    print("  khởi động emulator…", flush=True)
    sh(f"nohup {sdk}/emulator/emulator -avd phone -no-window -no-audio "
       f"-gpu swiftshader_indirect -no-snapshot -no-boot-anim -memory 3072 -cores 2 "
       f"> {ROOT}/emu.log 2>&1 &")
    sh("adb kill-server; adb start-server; sleep 2")
    sh("adb connect 127.0.0.1:5555")

else:
    # ── Không KVM: QEMU + android-x86_64 ISO ────────────────
    qp = ensure_qemu()
    ISO_NAME = "android-x86_64-9.0-r2.iso"
    iso = f"{ROOT}/{ISO_NAME}"
    ISO_URLS = [
        f"https://huggingface.co/datasets/sdafsdaf/android_x86_images/resolve/main/Android_09.0_Pie_API_28/x86_ISO/{ISO_NAME}",
        f"https://mirror.sjtu.edu.cn/android-x86/releases/9.0/r2/{ISO_NAME}",
    ]
    if not os.path.exists(iso) or os.path.getsize(iso) < 900_000_000:
        ok = False
        for u in ISO_URLS:
            try:
                print("↓ thử", u.split("/")[2], flush=True)
                dl(u, iso)
                if os.path.getsize(iso) > 900_000_000:
                    ok = True; break
                os.remove(iso)
            except Exception as e:
                print("   ✗", type(e).__name__, str(e)[:80], flush=True)
        if not ok:
            raise SystemExit("❌ Không tải được ISO. Hãy tải tay từ nguồn khác rồi đặt vào /content/android/")
    print("  ISO:", round(os.path.getsize(iso)/1e6), "MB", flush=True)

    print("  khởi động QEMU (TCG, mô phỏng mềm)…", flush=True)
    sh(f"nohup {qp}/qemu-system-x86_64 "
       f"-m 2048 -smp 2 -cpu max "
       f"-drive file={iso},media=cdrom,format=raw "
       f"-netdev user,id=net0,hostfwd=tcp::5555-:5555,hostfwd=tcp::6080-:6080 "
       f"-device virtio-net-pci,netdev=net0 "
       f"-vnc 127.0.0.1:1 -no-reboot "
       f"> {ROOT}/qemu.log 2>&1 &")
    time.sleep(10)
    sh("adb kill-server; adb start-server; sleep 2")
    sh("adb connect 127.0.0.1:5555")

print("\n  ⏳ đợi Android boot (có thể mất vài phút)…", flush=True)
if adb_ready():
    ip = sh(["adb","shell","getprop","ro.build.version.release"]).stdout.strip()
    print("✅ BOOT_OK — Android", ip)
else:
    print("❌ chưa boot được. Xem log:", ROOT)


## 🎮 CELL 4 · Bảng điều khiển trên điện thoại

Mình dựng một web app nhỏ (chạy trong chính notebook) làm 3 việc:

1. **Xem màn hình** — chụp frame từ Android qua `adb exec-out screencap`.
2. **Chạm** — chạm vào màn hình = `adb shell input tap`.
3. **Vuốt & phím** — swipe + nút BACK / HOME / VOLUME.

Chạy ô này, rồi mở đường dẫn mà Colab in ra trên **điện thoại**.

In [ ]:
# ══════════ CELL 4 · Web controller (điều khiển từ điện thoại) ══════════
import os, subprocess, threading, time, http.server, socketserver, urllib.parse, base64, io, json

ROOT = "/content/android"
os.chdir(ROOT)
PORT = 8080
FRAME = f"{ROOT}/frame.png"
STOP = threading.Event()

def sh(cmd):
    return subprocess.run(cmd, shell=isinstance(cmd,str), capture_output=True, text=True)

def adb(args, binary=False):
    cmd = ["adb"] + args
    r = subprocess.run(cmd, capture_output=True)
    return r.stdout if binary else r.stdout.decode(errors="replace")

# ── thread chụp màn hình liên tục ─────────────────────────
def capture_loop():
    while not STOP.is_set():
        try:
            data = subprocess.run(["adb","exec-out","screencap","-p"], capture_output=True, timeout=10).stdout
            if data[:4] == b"\x89PNG":
                tmp = FRAME + ".tmp"
                with open(tmp,"wb") as f: f.write(data)
                os.replace(tmp, FRAME)
        except Exception:
            pass
        time.sleep(0.25)   # ~4 fps

threading.Thread(target=capture_loop, daemon=True).start()

HTML = """<!doctype html><html><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1,user-scalable=no">
<title>Android @ Colab</title>
<style>
*{box-sizing:border-box;-webkit-tap-highlight-color:transparent}
body{margin:0;background:#0b0f14;color:#e6edf3;font:14px system-ui,sans-serif;
     display:flex;flex-direction:column;height:100vh;overflow:hidden}
#top{display:flex;gap:8px;align-items:center;padding:6px 8px;background:#161b22;flex:0 0 auto}
#top b{font-size:13px;margin-right:auto}
button{background:#238636;border:0;color:#fff;padding:8px 12px;border-radius:8px;font-size:14px}
button:active{filter:brightness(1.4)}
#stage{flex:1;position:relative;background:#000;overflow:hidden;display:flex;
       align-items:center;justify-content:center}
#img{max-width:100%;max-height:100%;object-fit:contain;display:block;touch-action:none}
#pad{position:absolute;inset:0}
</style></head><body>
<div id="top"><b>📱 Android · Colab</b>
<button onclick="key(4)">◀ Back</button>
<button onclick="key(3)">Home</button>
<button onclick="key(24)">Vol▲</button>
<button onclick="key(25)">Vol▼</button>
<button onclick="key(82)">Menu</button></div>
<div id="stage"><img id="img" src="/shot?t=1"><div id="pad"></div></div>
<script>
const img=document.getElementById('img'), pad=document.getElementById('pad');
function set(){img.src='/shot?t='+Date.now();}
setInterval(set,250);
function pos(e){const r=img.getBoundingClientRect();
  return [ (e.clientX-r.left)/r.width*img.naturalWidth,
           (e.clientY-r.top )/r.height*img.naturalHeight ];}
function send(u,b){fetch(u,{method:'POST',body:b});}
let last=null;
pad.addEventListener('touchstart',e=>{e.preventDefault();
  const t=e.touches[0];last=pos(t);},{passive:false});
pad.addEventListener('touchend',e=>{e.preventDefault();
  if(!last)return; const t=e.changedTouches[0], p=pos(t);
  if(Math.abs(p[0]-last[0])+Math.abs(p[1]-last[1])<18) send('/tap',`x=${p[0]}&y=${p[1]}`);
  else send('/swipe',`x1=${last[0]}&y1=${last[1]}&x2=${p[0]}&y2=${p[1]}&d=250`);
  last=null;},{passive:false});
pad.addEventListener('mousedown',e=>{last=pos(e);});
pad.addEventListener('mouseup',e=>{if(!last)return;const p=pos(e);
  send('/tap',`x=${p[0]}&y=${p[1]}`);last=null;});
function key(k){send('/key','k='+k);}
</script></body></html>"""

class H(http.server.BaseHTTPRequestHandler):
    def log_message(self, *a): pass
    def _send(self, code, ctype, body):
        self.send_response(code)
        self.send_header("Content-Type", ctype)
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Cache-Control", "no-store")
        self.end_headers()
        self.wfile.write(body)
    def do_GET(self):
        p = urllib.parse.urlparse(self.path).path
        if p == "/":
            self._send(200,"text/html; charset=utf-8", HTML.encode())
        elif p == "/shot":
            try:
                with open(FRAME,"rb") as f: d=f.read()
                self._send(200,"image/png", d)
            except Exception:
                self._send(404,"text/plain", b"no frame")
        else:
            self._send(404,"text/plain", b"404")
    def do_POST(self):
        n = int(self.headers.get("Content-Length",0))
        q = urllib.parse.parse_qs(self.rfile.read(n).decode())
        g = lambda k, d=0: float(q.get(k,[d])[0])
        p = urllib.parse.urlparse(self.path).path
        if p == "/tap":
            adb(["shell","input","tap",str(int(g("x"))),str(int(g("y")))])
        elif p == "/swipe":
            adb(["shell","input","swipe",str(int(g("x1"))),str(int(g("y1"))),
                 str(int(g("x2"))),str(int(g("y2"))),str(int(g("d",250)))])
        elif p == "/key":
            adb(["shell","input","keyevent",str(int(g("k")))])
        self._send(200,"text/plain", b"ok")

class S(socketserver.ThreadingMixIn, http.server.HTTPServer):
    daemon_threads = True
    allow_reuse_address = True

try:
    httpd = S(("0.0.0.0", PORT), H)
except OSError:
    PORT = 8081
    httpd = S(("0.0.0.0", PORT), H)
threading.Thread(target=httpd.serve_forever, daemon=True).start()
print("🎮 controller đang chạy ở cổng", PORT)


## 🔗 CELL 5 · Lấy đường dẫn để mở trên điện thoại

Chạy ô này → Colab in ra URL. **Mở URL đó bằng Chrome/Safari trên điện thoại** là chơi được.

> Nếu URL Colab không mở được từ mạng di động (một số nhà mạng chặn `googleusercontent.com`), chạy ô **5b** để lấy tunnel dự phòng.

In [ ]:
# ══════════ CELL 5 · Expose cổng ra Internet ══════════
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(PORT, path="/", height=760)

# ══════════ CELL 5b (dự phòng) · Tunnel qua SSH ══════════
# Chạy ô này nếu URL ở trên không mở được từ điện thoại.
# Nó in ra một URL https://... mới — mở URL đó trên điện thoại.
import subprocess, re, time
print("⏳ đang tạo tunnel…")
p = subprocess.Popen(
    ["ssh","-o","StrictHostKeyChecking=no","-o","ServerAliveInterval=30",
     "-R","80:localhost:%d"%PORT,"nokey@localhost.run"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
t0 = time.time()
while time.time()-t0 < 45 and url is None:
    line = p.stdout.readline()
    if not line: break
    m = re.search(r"https://[a-z0-9\-]+\.lhr\.life|https://[a-z0-9\-]+\.localhost\.run", line)
    if m: url = m.group(0)
    if line.strip(): print("  ", line.strip())
print("\n🌐 TUNNEL URL:", url if url else "(không lấy được — thử chạy lại ô này)")


## 📦 CELL 6 · Cài app APK lên máy Android

Bản **QEMU (không KVM)** dùng android-x86_64 **không có Play Store**, nên cài app bằng APK:

```python
!adb install /content/game.apk
```

Hoặc tải APK rồi cài:

```python
import urllib.request, subprocess
urllib.request.urlretrieve("https://link.apk", "/content/game.apk")
subprocess.run(["adb","install","-r","/content/game.apk"])
```

Bản **Emulator (có KVM)** dùng ảnh Google Play → **có Play Store**, cài app bình thường trên máy.


In [ ]:
# ══════════ CELL 6 · Tiện ích cài APK ══════════
def install_apk(path):
    r = subprocess.run(["adb","install","-r",path], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
    return r.returncode == 0

def shell(cmd):
    """Chạy lệnh trong Android, trả về output."""
    return subprocess.run(["adb","shell",cmd], shell=True, capture_output=True, text=True).stdout

print("Sẵn sàng. Ví dụ: install_apk('/content/game.apk')")
print("Danh sách app đang cài:", shell("pm list packages -3").count("package:") if True else 0, "app")


---

## 🧾 Ghi chú kỹ thuật

**Vì sao Colab/Kaggle không chơi game 3D nặng được**
- Không có `/dev/kvm` → QEMU phải mô phỏng mềm (TCG), chậm hơn ~10-20 lần.
- RAM ~12GB, CPU ~2-4 core dùng chung cho cả notebook.
- GPU của Colab (T4) **không** dùng được cho Android — Android trong QEMU vẽ bằng software renderer.

**Nên dùng khi nào**
- ✅ Test app Android nhanh, xem game chạy thế nào, chụp ảnh màn hình, dịch game
- ✅ Chơi game 2D nhẹ / game cũ (Lode Runner, Flappy, Tetris, game pixel)
- ❌ Game 3D nặng (Liên Minh, Genshin, PUBG) → cần VPS có KVM hoặc máy Android thật

**Nếu muốn chơi game nặng**: dùng `redroid` trên VPS có `/dev/kvm` (mình đã dựng thử được trên homelab của bạn — container boot OK, nhưng bạn không muốn mình đụng vào homelab nên mình đã dọn sạch).
